# Lung nodule detection: training and scan-level evaluation on Kaggle

This notebook trains a YOLO detector on LUNA16 and evaluates it **per scan**, the way the
LUNA16 challenge does, using the [lung-nodule-detection](https://github.com/jonahernan99/lung-nodule-detection)
package.

**Before running:**
1. *Settings → Accelerator*: choose a GPU. *Settings → Internet*: on.
2. *Add Input*: the dataset **vafaeii/luna16** ("LUNA16 (v3)"), which contains all ten subsets,
   `annotations.csv` and `annotations_excluded.csv`.
3. Leave `QUICK_RUN = True` for the first run (a few minutes). Set it to `False` for a full run.

In [ ]:
!pip install -q "lung-nodule-detection[train] @ git+https://github.com/jonahernan99/lung-nodule-detection.git"

## Settings

In [ ]:
from pathlib import Path

QUICK_RUN = True  # True: small smoke test. False: full training and evaluation.

MAX_SCANS = 20 if QUICK_RUN else None  # scans per split
EPOCHS = 10 if QUICK_RUN else 100
TEST_SUBSET, VAL_SUBSET = 9, 8
WORK = Path("/kaggle/working")

## Locate the LUNA16 files

In [ ]:
INPUT = Path("/kaggle/input")
# The dataset also ships a copy of annotations.csv inside evaluationScript/; take the top-level one.
ANNOTATIONS = min(INPUT.rglob("annotations.csv"), key=lambda p: len(p.parts))
LUNA_ROOT = ANNOTATIONS.parent
EXCLUDED = next(INPUT.rglob("annotations_excluded.csv"), None)
print("LUNA16 root:  ", LUNA_ROOT)
print("Annotations:  ", ANNOTATIONS)
print("Excluded:     ", EXCLUDED)

## Build the training data

Scans are split by LUNA16 subset, so every slice of a scan stays in one split. Test scans are
not exported as slices: they are evaluated on full volumes below.

In [ ]:
from lungnodule.dataset import build_dataset

summary = build_dataset(
    LUNA_ROOT,
    ANNOTATIONS,
    WORK / "dataset",
    test_subset=TEST_SUBSET,
    val_subset=VAL_SUBSET,
    max_scans_per_split=MAX_SCANS,
    progress=lambda uid, split, n: print(f"{split:<5} {n:>4} slices  {uid}"),
)
print(summary)

## Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov5su.pt")  # YOLOv5s backbone with Ultralytics' anchor-free head
model.train(
    data=str(WORK / "dataset" / "data.yaml"),
    epochs=EPOCHS,
    imgsz=512,
    batch=16,
    seed=0,
    project=str(WORK / "runs"),
    name="nodules",
    exist_ok=True,
)
WEIGHTS = WORK / "runs" / "nodules" / "weights" / "best.pt"
print("Weights:", WEIGHTS)

## Evaluate per scan

The detector runs on every slice of each test scan, detections are merged into 3D candidates,
and FROC/CPM is computed with false positives counted **per scan**, ignoring candidates on
LUNA16's excluded findings.

In [ ]:
from lungnodule.evaluation import evaluate_split
from lungnodule.inference import ultralytics_detector

detector = ultralytics_detector(str(WEIGHTS), confidence=0.01, image_size=512)
result = evaluate_split(
    LUNA_ROOT,
    ANNOTATIONS,
    WORK / "dataset" / "splits.csv",
    detector,
    WORK / "results",
    excluded_csv=EXCLUDED,
    max_scans=MAX_SCANS,
    progress=lambda uid, d, c: print(f"{d:>5} detections -> {c:>4} candidates  {uid}"),
)
print(f"Scans: {result.total_scans}, nodules: {result.total_nodules}")
for rate, sensitivity in zip(result.fp_rates, result.sensitivities):
    print(f"{rate:>6g} FP/scan  sensitivity {sensitivity:.3f}")
print(f"CPM: {result.cpm:.4f}")

## FROC curve

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(result.curve_fps_per_scan, result.curve_sensitivity, drawstyle="steps-post")
ax.scatter(result.fp_rates, result.sensitivities, zorder=3)
ax.set_xscale("log", base=2)
ax.set_xlim(0.1, 10)
ax.set_ylim(0, 1)
ax.set_xlabel("Average false positives per scan")
ax.set_ylabel("Sensitivity")
ax.set_title(f"Scan-level FROC on {result.total_scans} test scans (CPM {result.cpm:.3f})")
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig(WORK / "results" / "froc.png", dpi=150)
plt.show()

## Download the results

From the **Output** panel, download:
- `results/froc.json`: CPM, sensitivities and the full curve
- `results/candidates.csv`: detections in the LUNA16 submission format
- `results/froc.png`: the FROC curve
- `runs/nodules/results.png`: training curves

A quick run uses few scans, so its numbers only show that the pipeline works. Report results
from a full run.